# Pre-M0.7 — Joint IQ Power and Complex Magnitude

## Unit Objective

This notebook explores **power computation** of IQ signals from two equivalent perspectives: directly from real components (I and Q) and through the complex magnitude of the analytic signal. You will learn why both methods agree mathematically, how to compute per-example power, and how to visualize power distributions across a dataset.

## What You Will Learn

1. How to compute average power from I and Q components using $P = \text{mean}(I^2 + Q^2)$.
2. How to compute average power from the complex magnitude using $z = I + jQ$ and $P = \text{mean}(|z|^2)$.
3. The mathematical identity that guarantees both methods produce identical results.
4. How to compute per-example power and examine the distribution across a dataset.
5. How to visualize power for intuition building.

## IQ Data Contract

All IQ data in this notebook follows the canonical layout:

| Property | Value |
|----------|-------|
| Shape | `(N, 2, L)` |
| Axis 0 | Examples (N) |
| Axis 1 | I/Q (0 = I, 1 = Q) |
| Axis 2 | Time samples (L) |
| dtype | `np.float32` |
| SEED | 42 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Generating IQ Data

We create a synthetic dataset of N=8 IQ signals, each with L=64 time samples. The signals contain sinusoidal components with varying amplitudes and frequencies to produce diverse power levels.

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

N, L = 8, 64
t = np.linspace(0, 1, L, dtype=np.float32)

X = np.zeros((N, 2, L), dtype=np.float32)

for i in range(N):
    freq = 1.0 + i * 0.5
    amp = 0.5 + i * 0.25
    phase = rng.uniform(0, 2 * np.pi)
    X[i, 0, :] = amp * np.cos(2 * np.pi * freq * t + phase).astype(np.float32)
    X[i, 1, :] = amp * np.sin(2 * np.pi * freq * t + phase).astype(np.float32)

print(f"X.shape = {X.shape}")
print(f"X.dtype = {X.dtype}")
print(f"Example 0 — I range: [{X[0,0].min():.3f}, {X[0,0].max():.3f}]")
print(f"Example 0 — Q range: [{X[0,1].min():.3f}, {X[0,1].max():.3f}]")

## 2. Guided Development — Power from Real Components

The instantaneous power of a single sample is $p[n] = I^2[n] + Q^2[n]$. The average power over all time samples is:

$$P = \frac{1}{L} \sum_{n=0}^{L-1} \left( I^2[n] + Q^2[n] \right) = \text{mean}(I^2 + Q^2)$$

This is the **direct method** — no complex arithmetic needed.

In [ ]:
I = X[:, 0, :]  # shape (N, L)
Q = X[:, 1, :]  # shape (N, L)

instantaneous_power = I**2 + Q**2  # shape (N, L)
P_iq = np.mean(instantaneous_power, axis=1)  # shape (N,), per-example

print("Per-example power (direct I² + Q² method):")
for i in range(N):
    print(f"  Example {i}: P = {P_iq[i]:.6f}")

## 3. Guided Development — Power from Complex Magnitude

We can also view the IQ pair as a complex number $z = I + jQ$. The squared magnitude is:

$$|z|^2 = |I + jQ|^2 = I^2 + Q^2$$

Therefore the average power is:

$$P = \text{mean}(|z|^2)$$

This is the **complex method** — it uses NumPy's complex arithmetic.

In [ ]:
z = I + 1j * Q  # shape (N, L), complex128
z_mag_sq = np.abs(z)**2  # shape (N, L)
P_complex = np.mean(z_mag_sq, axis=1)  # shape (N,), per-example

print("Per-example power (complex magnitude method):")
for i in range(N):
    print(f"  Example {i}: P = {P_complex[i]:.6f}")

## 4. Why Both Methods Agree — Mathematical Proof

For any real numbers $I[n]$ and $Q[n]$:

$$|I[n] + jQ[n]|^2 = (\text{Re}(z))^2 + (\text{Im}(z))^2 = I^2[n] + Q^2[n]$$

This follows directly from the definition of complex magnitude: if $z = a + jb$, then $|z| = \sqrt{a^2 + b^2}$, so $|z|^2 = a^2 + b^2$.

Since the element-wise values are identical, their means must also be identical:

$$\text{mean}(I^2 + Q^2) = \text{mean}(|z|^2)$$

The identity holds for **every** element, not just the average — the two arrays `I**2 + Q**2` and `np.abs(z)**2` are numerically equal.

In [ ]:
elementwise_diff = np.abs(instantaneous_power - z_mag_sq)
print(f"Max element-wise difference: {elementwise_diff.max():.2e}")
print(f"All elements identical (within float32 tolerance): {np.allclose(instantaneous_power, z_mag_sq)}")

## 5. Per-Example Power Computation

Both methods naturally produce per-example power by reducing over axis 2 (time samples) while preserving axis 0 (examples). Let's visualize the per-example power values.

In [ ]:
examples = np.arange(N)
bar_width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(examples - bar_width/2, P_iq, bar_width, label='Direct (I² + Q²)', alpha=0.8)
bars2 = ax.bar(examples + bar_width/2, P_complex, bar_width, label='Complex (|z|²)', alpha=0.8)
ax.set_xlabel('Example Index')
ax.set_ylabel('Average Power')
ax.set_title('Per-Example Power: Direct vs Complex Method')
ax.set_xticks(examples)
ax.legend()
plt.tight_layout()
plt.show()

The bars overlap perfectly, confirming that both methods yield the same per-example power.

## 6. Power Distribution Across Examples

Let's also look at the distribution of power values across the entire dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(P_iq, bins=N, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Power Distribution (Direct Method)')
axes[0].set_xlabel('Average Power')
axes[0].set_ylabel('Count')

axes[1].hist(P_complex, bins=N, edgecolor='black', alpha=0.7, color='darkorange')
axes[1].set_title('Power Distribution (Complex Method)')
axes[1].set_xlabel('Average Power')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Direct method  — mean: {P_iq.mean():.6f}, std: {P_iq.std():.6f}")
print(f"Complex method — mean: {P_complex.mean():.6f}, std: {P_complex.std():.6f}")

## 7. Small Examples

### Example A: Single-sample power

Given a single IQ sample $(I, Q) = (3, 4)$, compute the power using both methods.

In [ ]:
i_val, q_val = 3.0, 4.0

P_direct = i_val**2 + q_val**2
z_val = i_val + 1j * q_val
P_mag = np.abs(z_val)**2

print(f"Direct:  P = {i_val}² + {q_val}² = {P_direct}")
print(f"Complex: |{z_val}|² = {P_mag}")
print(f"Match: {P_direct == P_mag}")

### Example B: Multi-sample average

Given three samples: $(I, Q) \in \{(1,0),\; (0,1),\; (1,1)\}$, compute the average power.

In [ ]:
i_arr = np.array([1.0, 0.0, 1.0])
q_arr = np.array([0.0, 1.0, 1.0])

P_direct = np.mean(i_arr**2 + q_arr**2)
z_arr = i_arr + 1j * q_arr
P_mag = np.mean(np.abs(z_arr)**2)

print(f"Direct:  P = {P_direct:.6f}")
print(f"Complex: P = {P_mag:.6f}")
print(f"Match: {np.isclose(P_direct, P_mag)}")

### Example C: Zero signal

If both I and Q are zero everywhere, the power must be zero.

In [ ]:
i_zero = np.zeros(10)
q_zero = np.zeros(10)

P_direct = np.mean(i_zero**2 + q_zero**2)
z_zero = i_zero + 1j * q_zero
P_mag = np.mean(np.abs(z_zero)**2)

print(f"Direct:  P = {P_direct}")
print(f"Complex: P = {P_mag}")
print(f"Both zero: {P_direct == 0.0 and P_mag == 0.0}")

## 8. Student Exercises

Complete the following exercises to solidify your understanding.

### Exercise 1: Custom IQ Power

Create an IQ signal where $I[n] = 2\cos(2\pi n / 8)$ and $Q[n] = 2\sin(2\pi n / 8)$ for $n = 0, \ldots, 15$. Compute the average power using both methods and verify they match.

In [ ]:
# STUDENT ATTEMPT
# Create I and Q arrays for n = 0..15
# Compute P_iq and P_complex
# Verify they match
pass

### Exercise 2: Power of a Noisy Signal

Generate $I[n] = 1 + 0.1 \cdot \text{noise}$ and $Q[n] = 0.5 + 0.1 \cdot \text{noise}$ for $L = 100$ samples using `rng.standard_normal`. Compute and compare both power methods.

In [ ]:
# STUDENT ATTEMPT
# Generate noisy I and Q
# Compute P_iq and P_complex
# Print and compare
pass

### Exercise 3: Axis Reduction

Given `X` of shape `(N, 2, L)`, compute the **total** average power across **all** examples (not per-example). What axis do you reduce over?

In [ ]:
# STUDENT ATTEMPT
# Compute total average power across all N examples
# Hint: you need to reduce over both axis 0 and axis 2
pass

### Optional Solution — Reveal Only After Attempt

In [ ]:
# Exercise 1 solution
n = np.arange(16, dtype=np.float32)
I_ex1 = 2.0 * np.cos(2 * np.pi * n / 8).astype(np.float32)
Q_ex1 = 2.0 * np.sin(2 * np.pi * n / 8).astype(np.float32)

P_ex1_direct = np.mean(I_ex1**2 + Q_ex1**2)
z_ex1 = I_ex1 + 1j * Q_ex1
P_ex1_complex = np.mean(np.abs(z_ex1)**2)

print(f"Exercise 1 — Direct: {P_ex1_direct:.6f}, Complex: {P_ex1_complex:.6f}, Match: {np.isclose(P_ex1_direct, P_ex1_complex)}")

In [ ]:
# Exercise 2 solution
L_ex2 = 100
I_ex2 = (1.0 + 0.1 * rng.standard_normal(L_ex2)).astype(np.float32)
Q_ex2 = (0.5 + 0.1 * rng.standard_normal(L_ex2)).astype(np.float32)

P_ex2_direct = np.mean(I_ex2**2 + Q_ex2**2)
z_ex2 = I_ex2 + 1j * Q_ex2
P_ex2_complex = np.mean(np.abs(z_ex2)**2)

print(f"Exercise 2 — Direct: {P_ex2_direct:.6f}, Complex: {P_ex2_complex:.6f}, Match: {np.isclose(P_ex2_direct, P_ex2_complex)}")

In [ ]:
# Exercise 3 solution
# Total average power: reduce over both examples (axis 0) and time (axis 2)
I_all = X[:, 0, :]  # (N, L)
Q_all = X[:, 1, :]  # (N, L)
total_power_direct = np.mean(I_all**2 + Q_all**2)
z_all = I_all + 1j * Q_all
total_power_complex = np.mean(np.abs(z_all)**2)

print(f"Exercise 3 — Total power: Direct: {total_power_direct:.6f}, Complex: {total_power_complex:.6f}")
print(f"Match: {np.isclose(total_power_direct, total_power_complex)}")

---

## PASS CRITERION CHALLENGES

The following three pass criterion challenges must **all** be satisfied to earn PASS status.

### PC-1 — INDEPENDENT AXIS EXPLANATION

Answer the following six questions about the IQ data axes. Write your answers in the Markdown cell below.

#### STUDENT AXIS EXPLANATION

#### STUDENT ATTEMPT

Write your answers here.

**Q1.** What does axis 0 represent in `X`?

**Q2.** What does axis 1 represent in `X`?

**Q3.** What does axis 2 represent in `X`?

**Q4.** If `X.shape == (8, 2, 64)`, how many examples are there?

**Q5.** If `X.shape == (8, 2, 64)`, how many time samples per example?

**Q6.** What is the dtype of `X` and why is it important for power computation?

In [ ]:
# STUDENT ATTEMPT — set to True after answering all 6 questions
AXES_EXPLANATION_VERIFIED = False  # Student/instructor must manually change to True
print(f"AXES_EXPLANATION_VERIFIED = {AXES_EXPLANATION_VERIFIED}")

#### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

**Q1.** Axis 0 represents the **example index** — each slice along axis 0 is one independent IQ signal.

**Q2.** Axis 1 represents the **I/Q channel** — index 0 is the in-phase component (I), index 1 is the quadrature component (Q).

**Q3.** Axis 2 represents the **time samples** — consecutive samples of the IQ signal over time.

**Q4.** There are **8** examples (axis 0 has size 8).

**Q5.** There are **64** time samples per example (axis 2 has size 64).

**Q6.** The dtype is `np.float32`. This is important because `float32` provides sufficient precision for IQ signal processing while being memory-efficient. Power computation (squaring and averaging) remains numerically stable in `float32` for typical signal amplitudes.

---

### PC-2 — INJECTED AXIS SWAP

A colleague accidentally transposed the I/Q and time axes in the data pipeline. The result is `X_swapped` with the wrong shape. You must diagnose the issue and recover the correct data.

In [ ]:
# Deliberately injected axis swap error
X_swapped = np.transpose(X, (0, 2, 1))  # shape (N, L, 2) — WRONG
print(f"X_swapped.shape = {X_swapped.shape}")
print(f"X_swapped.dtype = {X_swapped.dtype}")

#### STUDENT ATTEMPT

**Step 1.** Inspect `X_swapped.shape`. What is wrong?

**Step 2.** Which axis holds I/Q data in `X_swapped`?

**Step 3.** Why is this wrong for the IQ contract?

**Step 4.** Write the correction in the code cell below to produce `X_fixed`.

**Step 5.** Verify that `X_fixed` matches the original `X`.

In [ ]:
# STUDENT ATTEMPT
# Step 1-3: Write your explanation in the Markdown cell above
# Step 4: Write the correction below
# X_fixed = ...
# Step 5: Verify recovery
pass

#### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

**Step 1.** `X_swapped.shape == (N, L, 2)` — the last two axes are swapped compared to the contract `(N, 2, L)`.

**Step 2.** In `X_swapped`, axis 1 holds what should be the I/Q data (but it is mixed with time), and axis 2 holds what should be time samples.

**Step 3.** The contract requires axis 1 = I/Q (size 2) and axis 2 = time samples (size L). With the swap, axis 1 has size L and axis 2 has size 2, so any operation that slices `X[:, 0, :]` for I will grab the wrong data.

**Step 4-5.** The correction transposes back to restore the original axis order.

In [ ]:
# Solution
X_fixed = np.transpose(X_swapped, (0, 2, 1))

axis_swap_corrected = (
    X_fixed.shape == X.shape
    and X_fixed.dtype == X.dtype
    and np.array_equal(X_fixed, X)
)
assert X_fixed.shape == X.shape, f"Shape mismatch: {X_fixed.shape} != {X.shape}"
assert X_fixed.dtype == X.dtype, f"Dtype mismatch: {X_fixed.dtype} != {X.dtype}"
assert np.array_equal(X_fixed, X), "Values differ after correction"
print(f"Axis swap corrected: {axis_swap_corrected}")
print(f"X_fixed.shape = {X_fixed.shape}")

---

### PC-3 — IQ POWER AGREEMENT

Implement both power methods independently and verify that they agree within tight numerical tolerances.

In [ ]:
# STUDENT ATTEMPT
# Method A: Compute P_iq from I and Q components
# Method B: Compute P_complex from complex magnitude
# Verify they match with np.allclose using rtol=1e-5, atol=1e-7

POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

# P_iq = ...
# P_complex = ...
# power_consistency = ...
pass

#### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

In [ ]:
# Solution
POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

I_pc3 = X[:, 0, :]
Q_pc3 = X[:, 1, :]

# Method A: direct
P_iq = np.mean(I_pc3**2 + Q_pc3**2, axis=1)

# Method B: complex
z_pc3 = I_pc3 + 1j * Q_pc3
P_complex = np.mean(np.abs(z_pc3)**2, axis=1)

# Validate
power_consistency = np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
assert power_consistency, "Power methods disagree"
assert P_iq.shape == (N,), f"P_iq shape wrong: {P_iq.shape}"
assert P_complex.shape == (N,), f"P_complex shape wrong: {P_complex.shape}"

print(f"Power consistency: {power_consistency}")
print(f"P_iq      = {P_iq}")
print(f"P_complex = {P_complex}")
print(f"Max abs diff: {np.abs(P_iq - P_complex).max():.2e}")

---

## Automatic Validations

Run the following cell to verify all automatic checks.

In [ ]:
# Re-run validations for PC-2 and PC-3
X_swapped_check = np.transpose(X, (0, 2, 1))
X_fixed_check = np.transpose(X_swapped_check, (0, 2, 1))

axis_swap_corrected = (
    X_fixed_check.shape == X.shape
    and X_fixed_check.dtype == X.dtype
    and np.array_equal(X_fixed_check, X)
)
assert X_fixed_check.shape == X.shape
assert X_fixed_check.dtype == X.dtype
assert np.array_equal(X_fixed_check, X)

I_val = X[:, 0, :]
Q_val = X[:, 1, :]
P_iq_val = np.mean(I_val**2 + Q_val**2, axis=1)
z_val = I_val + 1j * Q_val
P_complex_val = np.mean(np.abs(z_val)**2, axis=1)

POWER_RTOL = 1e-5
POWER_ATOL = 1e-7
power_consistency = np.allclose(P_iq_val, P_complex_val, rtol=POWER_RTOL, atol=POWER_ATOL)
assert power_consistency

print("All automatic validations PASSED.")

---

## Manual Evaluation of Axis Explanation

The instructor (or student self-check) must verify that the **STUDENT ATTEMPT** answers in PC-1 are correct.

| Question | Correct Answer |
|----------|----------------|
| Q1 | Axis 0 = examples |
| Q2 | Axis 1 = I/Q (0=I, 1=Q) |
| Q3 | Axis 2 = time samples |
| Q4 | 8 examples |
| Q5 | 64 time samples |
| Q6 | `float32`, sufficient precision, memory-efficient |

**Action required:** After reviewing the student's answers, set `AXES_EXPLANATION_VERIFIED = True` in the PC-1 code cell above if all answers are correct.

---

## PASS CRITERION GATE

In [ ]:
# PC-1: Axis explanation verified
pc1_pass = AXES_EXPLANATION_VERIFIED
print(f"PC-1 (Axis Explanation):    {'PASS' if pc1_pass else 'WAIT'}")

# PC-2: Axis swap corrected
X_swapped_final = np.transpose(X, (0, 2, 1))
X_fixed_final = np.transpose(X_swapped_final, (0, 2, 1))
pc2_pass = (
    X_fixed_final.shape == X.shape
    and X_fixed_final.dtype == X.dtype
    and np.array_equal(X_fixed_final, X)
)
print(f"PC-2 (Axis Swap Corrected): {'PASS' if pc2_pass else 'WAIT'}")

# PC-3: Power agreement
pc3_pass = bool(power_consistency)
print(f"PC-3 (Power Agreement):     {'PASS' if pc3_pass else 'WAIT'}")

# Final status
all_pass = pc1_pass and pc2_pass and pc3_pass
final_status = "PASS" if all_pass else "WAIT"
print(f"\nPRE-M0.7 FINAL STATUS: {final_status}")